# 🏗️ Day 04a: sklearn Pipelines — Production-Ready ML

---

## 🎯 What You'll Master Today
- Pipeline & make_pipeline
- ColumnTransformer for mixed types
- Custom transformers
- Pipeline serialization (joblib)

**Goal:** Build a complete end-to-end pipeline from memory.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  PIPELINE — Why?                                                 ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Without pipeline:                                               ║
║    scaler.fit_transform(X_train) → forget on X_test? 💀        ║
║    encoder.fit_transform(X_train) → wrong order? 💀            ║
║    model.fit(X_train) → debugging nightmare 💀                  ║
║                                                                   ║
║  With pipeline:                                                  ║
║    pipe.fit(X_train, y_train)      ← one call, all steps       ║
║    pipe.predict(X_test)            ← no leakage possible        ║
║    pipe.score(X_test, y_test)      ← clean evaluation          ║
║                                                                   ║
║  Pipeline guarantees:                                            ║
║    ✓ No data leakage (fit only on train)                       ║
║    ✓ Reproducible (same steps every time)                      ║
║    ✓ Serializable (save/load with joblib)                      ║
║    ✓ Grid-searchable (tune any step's params)                  ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin

# ============================================================
# 1. Basic Pipeline — named steps
# ============================================================

from sklearn.datasets import load_breast_cancer
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Pipeline with named steps
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000))
])

# One call does everything
pipe.fit(X_train, y_train)
score = pipe.score(X_test, y_test)
print(f"Pipeline score: {score:.4f}")

# make_pipeline — auto-names steps
pipe2 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
pipe2.fit(X_train, y_train)
print(f"make_pipeline score: {pipe2.score(X_test, y_test):.4f}")

# Access steps
print(f"\nSteps: {[name for name, _ in pipe.steps]}")
print(f"Scaler mean (first 3): {pipe['scaler'].mean_[:3].round(4)}")

In [ ]:
# ============================================================
# 2. ColumnTransformer — Mixed Feature Types (THE REAL DEAL)
# ============================================================

# Create realistic mixed-type dataset
np.random.seed(42)
n = 500
df = pd.DataFrame({
    'age': np.random.randint(18, 70, n),
    'salary': np.random.exponential(50000, n),
    'experience': np.random.randint(0, 30, n).astype(float),
    'city': np.random.choice(['NYC', 'LA', 'Chicago', 'Houston'], n),
    'education': np.random.choice(['BS', 'MS', 'PhD'], n),
})
# Add some NaN
df.loc[np.random.choice(n, 30, replace=False), 'experience'] = np.nan
df.loc[np.random.choice(n, 20, replace=False), 'salary'] = np.nan

y = (df['salary'].fillna(df['salary'].median()) > 50000).astype(int)
X = df.drop(columns=[])

print(f"Features: {list(df.columns)}")
print(f"Missing: {df.isnull().sum().to_dict()}")
print(f"Dtypes: {df.dtypes.to_dict()}\n")

# Define column groups
numeric_features = ['age', 'salary', 'experience']
categorical_features = ['city', 'education']

# Numeric: impute → scale
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

# Categorical: impute → one-hot
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False)),
])

# Combine with ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features),
])

# Full pipeline: preprocess → model
full_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Split and evaluate
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
full_pipe.fit(X_train, y_train)
print(f"Full pipeline score: {full_pipe.score(X_test, y_test):.4f}")

# CV with pipeline — no leakage!
cv_scores = cross_val_score(full_pipe, X, y, cv=5, scoring='accuracy')
print(f"CV scores: {cv_scores.round(4)} → Mean: {cv_scores.mean():.4f}")

In [ ]:
# ============================================================
# 3. GridSearch on Pipeline — tune ANY step
# ============================================================

# Access params with double underscore: step__param
param_grid = {
    'preprocessor__num__imputer__strategy': ['mean', 'median'],
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [5, 10, None],
}

grid = GridSearchCV(
    full_pipe, param_grid, cv=3, scoring='accuracy', n_jobs=-1
)
grid.fit(X_train, y_train)

print(f"Best params: {grid.best_params_}")
print(f"Best CV score: {grid.best_score_:.4f}")
print(f"Test score: {grid.score(X_test, y_test):.4f}")

print("\n💡 Naming: pipeline_step__nested_step__parameter")
print("   Example: preprocessor__num__imputer__strategy")

In [ ]:
# ============================================================
# 4. Custom Transformer — plug into Pipeline
# ============================================================

class LogTransformer(BaseEstimator, TransformerMixin):
    """Apply log1p to specified columns."""
    def __init__(self, columns=None):
        self.columns = columns
    
    def fit(self, X, y=None):
        return self  # nothing to learn
    
    def transform(self, X):
        X = X.copy()
        if self.columns is not None:
            for col in self.columns:
                if isinstance(X, pd.DataFrame):
                    X[col] = np.log1p(X[col].clip(lower=0))
                else:
                    X[:, col] = np.log1p(np.clip(X[:, col], 0, None))
        else:
            X = np.log1p(np.clip(X, 0, None))
        return X

# Use in pipeline
custom_pipe = Pipeline([
    ('log', LogTransformer(columns=['salary'])),
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000)),
])

custom_pipe.fit(X_train, y_train)
print(f"Custom transformer pipeline: {custom_pipe.score(X_test, y_test):.4f}")

print("\n💡 Custom transformer rules:")
print("  1. Inherit BaseEstimator + TransformerMixin")
print("  2. __init__ stores params (no computation!)")
print("  3. fit() learns from data (or just returns self)")
print("  4. transform() applies the transformation")

In [ ]:
# ============================================================
# 5. Pipeline Serialization — Save & Load
# ============================================================
import joblib
import os
import tempfile

# Save the full pipeline (preprocessor + model)
save_path = os.path.join(tempfile.gettempdir(), 'model_pipeline.joblib')
joblib.dump(full_pipe, save_path)
print(f"Saved to: {save_path}")
print(f"File size: {os.path.getsize(save_path) / 1024:.1f} KB")

# Load and predict
loaded_pipe = joblib.load(save_path)
score_loaded = loaded_pipe.score(X_test, y_test)
print(f"Loaded pipeline score: {score_loaded:.4f}")

# Predict on new data — just pass raw data
new_data = pd.DataFrame({
    'age': [30],
    'salary': [60000],
    'experience': [5],
    'city': ['NYC'],
    'education': ['MS'],
})
prediction = loaded_pipe.predict(new_data)
print(f"\nPrediction for new data: {prediction}")

# Cleanup
os.remove(save_path)

print("\n💡 joblib.dump(pipeline) saves EVERYTHING:")
print("   scaler params, encoder mappings, model weights")
print("   Just load + predict on raw data. No preprocessing needed.")

In [ ]:
# ============================================================
# 6. Production Pipeline Template — MEMORIZE THIS
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

# --- Step 1: Define column groups ---
num_cols = ['age', 'salary', 'experience']
cat_cols = ['city', 'education']

# --- Step 2: Build sub-pipelines ---
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')),
])

# --- Step 3: ColumnTransformer ---
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols),
])

# --- Step 4: Full pipeline ---
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42)),
])

# --- Step 5: Train & evaluate ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pipeline.fit(X_train, y_train)
print(f"Test score: {pipeline.score(X_test, y_test):.4f}")
print(f"CV score: {cross_val_score(pipeline, X, y, cv=5).mean():.4f}")

print("\n🏗️ This 5-step template covers 90% of ML pipeline needs!")

---

## 🗺️ Pipeline Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  PIPELINE — Component Guide                                     ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Pipeline              → chain steps sequentially               ║
║  make_pipeline          → auto-named Pipeline                   ║
║  ColumnTransformer      → different transforms per column type  ║
║  BaseEstimator +                                                ║
║    TransformerMixin     → custom transformer class              ║
║  FunctionTransformer    → quick lambda-based transformer        ║
║                                                                   ║
║  PARAM ACCESS (for GridSearch):                                  ║
║    step__param                → pipeline_step__parameter        ║
║    step__nested__param        → nested pipeline parameter       ║
║                                                                   ║
║  SAVE/LOAD:                                                      ║
║    joblib.dump(pipe, path)    → serialize everything            ║
║    joblib.load(path)          → load and predict raw data       ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Why use Pipeline? | Prevents data leakage, reproducible, serializable, grid-searchable |
| 2 | Pipeline vs ColumnTransformer? | Pipeline = sequential steps. ColumnTransformer = different transforms per column group |
| 3 | How to tune pipeline params? | Use double underscore: step__param. GridSearchCV works directly on pipeline |
| 4 | How to write a custom transformer? | Inherit BaseEstimator + TransformerMixin. Implement fit() and transform() |
| 5 | How to deploy an ML model? | joblib.dump(pipeline) → load in production → pass raw data to predict() |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Pipeline chains steps — fit/predict does everything  │
## │  • ColumnTransformer for mixed types (num + cat)        │
## │  • Custom: BaseEstimator + TransformerMixin             │
## │  • GridSearch: step__param double underscore access     │
## │  • joblib.dump() saves the entire pipeline              │
## │  • 5-step template handles 90% of ML projects          │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Deep Learning Fundamentals** — neural net math from scratch